# Этап 1: Сбор метаданных
- Только cs.CL статьи
- За всё время

In [1]:
import arxiv
import pandas as pd
import calendar
import time
import os
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

In [2]:
# ===== КОНФИГУРАЦИЯ =====
CAT = 'cs.CL'
START_YEAR = 1991
CURRENT_YEAR = datetime.now().year
TEMP_DIR = Path('../../data/metadata/temp_years')
OUTPUT_DIR = Path('../../data/metadata')
FINAL_FILE = f"arxiv_NLP_{START_YEAR}_{CURRENT_YEAR}_metadata.csv"

TEMP_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = arxiv.Client(
    page_size=1000,
    delay_seconds=5,
    num_retries=5
)

In [3]:
print(f"Начинаем парсинг категории {CAT} с {START_YEAR} по {CURRENT_YEAR} год.")

total_papers_count = 0

for year in range(START_YEAR, CURRENT_YEAR + 1):
    year_papers = []
    year_start_time = time.time()
    
    checkpoint_file = TEMP_DIR / f"arxiv_{CAT}_{year}.csv"
    if checkpoint_file.exists():
        print(f"Год {year} уже скачан ({checkpoint_file.name}). Пропускаем.")
        continue

    print(f"\nProcessing {year}...")
    
    for month in tqdm(range(1, 13), desc=f"Year {year}", leave=False):
        if year == CURRENT_YEAR and month > datetime.now().month:
            break

        last_day = calendar.monthrange(year, month)[1]
        
        start_str = f"{year}{month:02d}01000000"
        end_str = f"{year}{month:02d}{last_day}235959"
        
        query = f"cat:{CAT} AND submittedDate:[{start_str} TO {end_str}]"
        
        try:
            search = arxiv.Search(
                query=query,
                sort_by=arxiv.SortCriterion.SubmittedDate
            )
            
            # Генератор результатов
            results = client.results(search)
            
            for paper in results:
                if paper.primary_category == CAT:
                    year_papers.append({
                        'arxiv_id': paper.get_short_id(),
                        'title': paper.title,
                        'authors': ', '.join([a.name for a in paper.authors]),
                        'summary': paper.summary.replace('\n', ' '),
                        'primary_category': paper.primary_category,
                        'published': paper.published.strftime('%Y-%m-%d'),
                        'updated': paper.updated.strftime('%Y-%m-%d'),
                        'entry_id': paper.entry_id
                    })
        except Exception as e:
            print(f"Ошибка при парсинге {year}-{month:02d}: {e}")
            time.sleep(10)

    # === СОХРАНЕНИЕ ЧЕКПОИНТА ===
    if year_papers:
        df_year = pd.DataFrame(year_papers)
        df_year['html_url'] = 'https://arxiv.org/html/' + df_year['arxiv_id'].astype(str)
        df_year.to_csv(checkpoint_file, index=False, encoding='utf-8')
        total_papers_count += len(df_year)
        print(f"Год {year} завершён. Сохранено {len(df_year)} статей. (Время: {time.time() - year_start_time:.1f}с)")
    else:
        print(f"В {year} году статей не найдено.")

print(f"\nПарсинг завершён. Начинаем объединение файлов...")

# ===== ОБЪЕДИНЕНИЕ ВСЕХ ФАЙЛОВ =====
all_dfs = []
csv_files = sorted(TEMP_DIR.glob(f"arxiv_{CAT}_*.csv"))

if csv_files:
    print(f"Найдено {len(csv_files)} файлов для объединения.")
    for file in tqdm(csv_files, desc="Merging"):
        all_dfs.append(pd.read_csv(file))
    
    full_df = pd.concat(all_dfs, ignore_index=True)
    full_path = OUTPUT_DIR / FINAL_FILE
    full_df.to_csv(full_path, index=False, encoding='utf-8')
    
    print(f"\nSUCCESS! Итоговый датасет сохранён в: {full_path}")
    print(f"Всего статей: {len(full_df)}")
    print(f"Размер в памяти: {full_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
else:
    print("Файлы с данными не найдены.")

Начинаем парсинг категории cs.CL с 1991 по 2026 год.

Processing 1991...


В 1991 году статей не найдено.

Processing 1992...


В 1992 году статей не найдено.

Processing 1993...


В 1993 году статей не найдено.

Processing 1994...


Год 1994 завершён. Сохранено 223 статей. (Время: 60.9с)

Processing 1995...


...

Год 2024 завершён. Сохранено 14999 статей. (Время: 217.3с)

Processing 2025...


Год 2025 завершён. Сохранено 17083 статей. (Время: 254.3с)

Processing 2026...


Год 2026 завершён. Сохранено 1845 статей. (Время: 37.7с)

Парсинг завершён. Начинаем объединение файлов...
Найдено 33 файлов для объединения.


Merging: 100%|█████████████████████████████████████████████████████████████████████████| 33/33 [00:02<00:00, 13.02it/s]



SUCCESS! Итоговый датасет сохранён в: ..\..\data\metadata\arxiv_NLP_1991_2026_metadata.csv
Всего статей: 77112
Размер в памяти: 140.72 MB
